# Paper 2 — Policy Scenario Analysis
**Targeting Logic as a Governance Variable: Agent-Based Evidence from UK Residential Retrofit**

**Target journal:** *Policy Studies Journal* (Research Note, ≤5,000 words)

---

## Research question

Residential retrofit programmes must be targeted: limited budgets mean not every home can
receive a heat pump. But different institutional actors — energy suppliers, local authorities,
network operators — hold different data about households, so each actor can only implement
the strategy their information set supports. This creates a **polycentric coordination problem**
(Ostrom 1990; Feiock 2013): the targeting logic that maximises aggregate demand reduction is
not the same as the one that maximises equity, and neither can be achieved by any single actor
acting alone.

This notebook quantifies the efficiency–equity tradeoff by running three targeting scenarios
through the Newcastle MABM (97,714 dwellings), each representing what a specific institutional
actor could implement with the data they actually hold:

| Scenario | Institutional actor | Information set | Governance arrangement |
|---|---|---|---|
| `hp_top_users` | Energy supplier | Smart-meter consumption history | Unilateral |
| `hp_fuel_poverty` | Local authority | EPC E/F/G + bottom 2 income quintiles | Unilateral |
| `hp_composite` | Both coordinated | Consumption + EPC + income, scored | Bilateral / polycentric |

All three scenarios receive the **same programme budget** (`PROGRAMME_HOMES` installations).
Fixing the budget means the comparison isolates targeting logic from scale — the only thing
that differs is which homes get selected.

The gap between `hp_composite` and the two unilateral scenarios is the **coordination dividend**:
what is achievable only when actors share data. This is the paper's core empirical claim.

A fourth scenario — `hp_grid_constrained` (network operator targeting by substation headroom)
— is planned for a future extension once NCC grid data are available.

## Relationship to other notebooks

- `sensitivity_analysis.ipynb` — produces the crossover curve (how the gap between scenarios
  shifts as budget scales up) and calibration parameter sensitivity for Paper 1. The crossover
  curve from that notebook is Figure 5 in Paper 2; it is not reproduced here.
- `policy_scenarios_summary.ipynb` — the earlier two-scenario comparison (`hp_social_rent` vs
  `hp_top_users`); this notebook supersedes it for Paper 2 purposes.

## Runtime
Default is **fast annualization** (`USE_FAST_ANNUALIZATION = True`): simulate 28 days and scale
to annual equivalent. This is fast enough for exploration (~5–10 min on 8 cores). Set
`USE_FAST_ANNUALIZATION = False` for paper-quality figures (true annual, ~2–3 hours).

## Imports

Standard scientific stack plus `household_energy.model.EnergyModel` (the MABM).
`tempfile` and `yaml` are used to pass config overrides to the model without modifying
the default YAML on disk — each scenario run gets its own temporary config file.

In [ ]:
from __future__ import annotations

from pathlib import Path
import multiprocessing as mp
import hashlib
import random
import tempfile
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import yaml

from household_energy.model import EnergyModel

warnings.filterwarnings('ignore', message='.*GeoSeries.notna.*')
warnings.filterwarnings('ignore', category=FutureWarning)

## 1) Settings

### Key parameters to tune

**`PROGRAMME_HOMES`** — the number of heat-pump installations the programme budget covers.
Setting this to a fixed number (rather than uptake rates) is deliberate: it isolates targeting
logic from scale, making the comparison interpretable. 2,000 is roughly 2% of the Newcastle
stock and within the range of credible 5-year LA retrofit targets.

**`HP_COST_GBP`** — assumed all-in installation cost per heat pump (£12,000 is the mid-range
estimate from the Heat Pump Association 2023). Used only to compute cost-per-GWh; it cancels
out in scenario comparisons since all scenarios have the same number of installations.

**`COMPOSITE_WEIGHTS`** (defined in Section 3) — the 40/40/20 split between consumption, EPC
vulnerability, and income vulnerability. These are a design choice; the paper should report
sensitivity to alternative weights (planned as a short robustness addition to this notebook).

### Colour palette
Consistent with `too_hot_to_handle_figures.ipynb` and `sensitivity_analysis.ipynb`.

In [ ]:
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

GEOJSON  = ROOT / 'data' / 'epc_abm_newcastle.geojson'
CLIMATE  = ROOT / 'data' / 'ncc_2t_timeseries_2010_2039.parquet'
HIDP_CSV = ROOT / 'data' / 'hidp_uprn_matches_tiered.csv'
OUTDIR   = ROOT / 'notebooks' / 'results' / 'paper2_policy'
OUTDIR.mkdir(parents=True, exist_ok=True)

# Fixed programme budget — all three scenarios treat exactly this many homes.
PROGRAMME_HOMES = 2000

# Heat-pump cost assumption for cost-per-GWh (HPA 2023 mid estimate).
HP_COST_GBP = 12_000

# Runtime
USE_FAST_ANNUALIZATION = True   # False for paper-quality figures
WINDOW_DAYS = 28
START_UTC   = '2023-01-01T00:00:00Z'
N_PROCS     = max(1, min(8, (mp.cpu_count() or 2) - 1))

WINDOW_HOURS  = (WINDOW_DAYS * 24) if USE_FAST_ANNUALIZATION else (365 * 24)
ANNUAL_FACTOR = (365.0 / WINDOW_DAYS) if USE_FAST_ANNUALIZATION else 1.0

PROCESS_COLUMN = 'lsoa_code'  # parallel shard unit
AREA_COLUMN    = 'ward_code'  # reporting geography

# Colour palette (shared across all paper notebooks)
BLUE   = '#4c78a8'
ORANGE = '#f58518'
GREEN  = '#54a24b'
RED    = '#e45756'
GRAY_M = '#9e9e9e'

SCENARIO_COLOURS = {
    'hp_top_users':    BLUE,
    'hp_fuel_poverty': GREEN,
    'hp_composite':    ORANGE,
}
SCENARIO_LABELS = {
    'hp_top_users':    'Top users\n(energy supplier)',
    'hp_fuel_poverty': 'Fuel poverty\n(local authority)',
    'hp_composite':    'Composite\n(coordinated)',
}
SCENARIOS = ['hp_top_users', 'hp_fuel_poverty', 'hp_composite']

plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        150,
})

print(f'Programme size:       {PROGRAMME_HOMES:,} homes')
print(f'HP cost assumption:   £{HP_COST_GBP:,}/installation')
print(f'Total programme cost: £{PROGRAMME_HOMES * HP_COST_GBP / 1e6:.1f}m')
print(f'Window hours:         {WINDOW_HOURS:,} | Annual factor: {ANNUAL_FACTOR:.3f}')
print(f'Parallel workers:     {N_PROCS}')
print(f'Outputs →             {OUTDIR.resolve()}')

# ── Calibration auto-pickup ──────────────────────────────────────────────────
# Find newest results/calibration_<timestamp>/calibrated_config.yaml and use it
# as the base config for every EnergyModel run. Scenario-specific overrides
# (e.g. heatpump_adoption_rate) are deep-merged on top below.
_cal_dirs = sorted((ROOT / 'results').glob('calibration_*'), key=lambda p: p.name)
if not _cal_dirs:
    raise FileNotFoundError(f"No calibration_* directory under {ROOT / 'results'}")
CAL_DIR = _cal_dirs[-1]
CAL_YAML = CAL_DIR / 'calibrated_config.yaml'
if not CAL_YAML.exists():
    raise FileNotFoundError(f"Calibrated config not found: {CAL_YAML}")
with CAL_YAML.open() as _f:
    CALIBRATED_CFG: dict = yaml.safe_load(_f) or {}
print(f'Calibration base: {CAL_DIR.name}')


def _deep_merge(base: dict, override: dict) -> dict:
    """Recursive dict merge (override wins on conflict)."""
    out = dict(base)
    for k, v in (override or {}).items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = _deep_merge(out[k], v)
        else:
            out[k] = v
    return out


## 2) Load and enrich synthetic population

The synthetic population is stored across two files:

- **`epc_abm_newcastle.geojson`** — one row per dwelling (UPRN). Contains EPC-derived attributes:
  `sap_band_ord` (1=G, 7=A), `floor_area_m2`, `property_type`, `property_age`,
  `retrofit_envelope_score`, `energy_cal_kwh` (calibrated baseline demand).

- **`hidp_uprn_matches_tiered.csv`** — HIDP (Household Income and Demographics Profile)
  matched to UPRN. Contains `tenure`, `hh_income_band` (q1_lowest–q5_highest),
  `schedule_type` (occupancy pattern), `hh_children`, `hh_edu_detail`.

The merge is left-join on UPRN — dwellings without a HIDP match keep their EPC attributes
but have NaN for sociodemographic fields. These homes are still simulated but drop out of
income-band analyses.

In [ ]:
def load_enriched_gdf(geojson_path: Path, hidp_csv_path: Path | None) -> gpd.GeoDataFrame:
    g = gpd.read_file(geojson_path)
    g['UPRN'] = g['UPRN'].astype(str).str.strip()

    if hidp_csv_path and hidp_csv_path.exists():
        hidp = pd.read_csv(hidp_csv_path, low_memory=False)
        hidp.columns = [c.strip() for c in hidp.columns]
        hidp['uprn_chr'] = hidp['uprn_chr'].astype(str).str.strip()
        hidp = hidp.drop_duplicates(subset=['uprn_chr'])
        g = g.merge(hidp, how='left', left_on='UPRN', right_on='uprn_chr', suffixes=('_geo', '_hidp'))

        for base in ['lsoa_code', 'ward_code', 'local_authority']:
            geo_col, hidp_col = f'{base}_geo', f'{base}_hidp'
            if base not in g.columns and (geo_col in g.columns or hidp_col in g.columns):
                if geo_col in g.columns and hidp_col in g.columns:
                    g[base] = g[geo_col].combine_first(g[hidp_col])
                elif geo_col in g.columns:
                    g[base] = g[geo_col]
                else:
                    g[base] = g[hidp_col]
    return g


def to_wgs84_points(gdf_in: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Convert any geometry type to WGS84 point (centroid if polygon)."""
    g = gdf_in.dropna(subset=['geometry']).copy()
    g = g.to_crs(4326) if g.crs else g.set_crs(4326)
    if (g.geometry.geom_type != 'Point').any():
        g['geometry'] = g.geometry.centroid
    return g


gdf_all   = load_enriched_gdf(GEOJSON, HIDP_CSV if HIDP_CSV.exists() else None)
gdf_focus = to_wgs84_points(gdf_all.copy())
gdf_focus['AgentID'] = gdf_focus['UPRN'].astype(str)

selected_units = (
    gdf_focus[PROCESS_COLUMN].astype(str).replace('', np.nan).dropna().unique().tolist()
)

print(f'Total dwellings: {len(gdf_focus):,}')
print(f'LSOAs (process shards): {len(selected_units):,}')

# Quick check on HIDP match rate
n_with_income = gdf_focus['hh_income_band'].notna().sum() if 'hh_income_band' in gdf_focus.columns else 0
print(f'Dwellings with income band: {n_with_income:,} ({n_with_income/len(gdf_focus)*100:.1f}%)')

## 3) Scenario definitions: composite scoring and policy masks

### Why fixed budget rather than uptake rates?

The earlier `policy_scenarios_summary.ipynb` ran `hp_social_rent` at 50% uptake and
`hp_top_users` at 20% uptake. This confounds targeting logic with scale: a larger share of
a bigger eligible pool means more homes treated, which mechanically increases aggregate savings
regardless of targeting quality. Fixing `PROGRAMME_HOMES` removes this ambiguity.

### Composite score construction

The composite score represents information a coordinated programme could combine from the
energy supplier (consumption) and local authority (EPC, income):

```
score = 0.40 × consumption_norm   (kWh/yr, normalised to [0,1])
      + 0.40 × epc_vulnerability   ((8 − sap_band_ord) / 7, so G=1.0, A=0.0)
      + 0.20 × income_vulnerability (q1_lowest=1.0 … q5_highest=0.0)
```

The consumption weight prioritises homes where a heat pump will actually reduce demand;
the EPC weight prioritises homes with the worst fabric (greatest retrofit need);
the income weight gives a modest equity nudge without dominating the selection.
A sensitivity check with alternative weights is planned (see roadmap).

### `hp_fuel_poverty` eligible pool

EPC E/F/G (`sap_band_ord ≤ 3`) AND bottom 2 income quintiles — this is the Warm Homes
Plan eligibility criterion. Within the eligible pool, homes are ranked by consumption
so the LA prioritises those where the heat pump saves the most (defensible within the
equity constraint). If the eligible pool is smaller than `PROGRAMME_HOMES`, all eligible
homes are treated and the shortfall is reported.

In [ ]:
META_COLS = [
    'AgentID', 'UPRN', 'geometry', AREA_COLUMN, PROCESS_COLUMN,
    'property_type', 'tenure', 'hh_income_band', 'schedule_type',
    'sap_band_ord', 'energy_cal_kwh',
]
for col in META_COLS:
    if col != 'geometry' and col not in gdf_focus.columns:
        gdf_focus[col] = np.nan

# Consumption (energy supplier's information)
cons_raw  = pd.to_numeric(gdf_focus.get('energy_cal_kwh', 0), errors='coerce').fillna(0.0)
cons_norm = (cons_raw - cons_raw.min()) / (cons_raw.max() - cons_raw.min() + 1e-9)

# EPC vulnerability (MHCLG open data — available to LA)
# sap_band_ord: 1=G (worst), 7=A (best). Invert so G→1.0, A→0.0.
epc_ord  = pd.to_numeric(gdf_focus.get('sap_band_ord', 4), errors='coerce').fillna(4)
epc_vuln = (8 - epc_ord) / 7.0

# Income vulnerability (HIDP — available to LA via means-test)
INCOME_VULN_MAP = {
    'q1_lowest':  1.00,
    'q2_low':     0.75,
    'q3_mid':     0.50,
    'q4_high':    0.25,
    'q5_highest': 0.00,
}
inc_band = gdf_focus.get('hh_income_band', pd.Series('', index=gdf_focus.index)).astype(str)
inc_vuln = inc_band.map(INCOME_VULN_MAP).fillna(0.50)  # missing → mid

# Composite score (coordinated, bilateral data sharing)
COMPOSITE_WEIGHTS = {'consumption': 0.40, 'epc': 0.40, 'income': 0.20}
composite_score = (
    COMPOSITE_WEIGHTS['consumption'] * cons_norm
    + COMPOSITE_WEIGHTS['epc']         * epc_vuln
    + COMPOSITE_WEIGHTS['income']      * inc_vuln
)
gdf_focus['composite_score'] = composite_score

# --- Build POLICY_MASKS: which homes get treated under each scenario ---

# hp_top_users: energy supplier picks highest-consumption homes
top_users_idx = cons_raw.nlargest(PROGRAMME_HOMES).index

# hp_fuel_poverty: LA targets EPC E/F/G AND bottom 2 income quintiles
fp_eligible = epc_ord.le(3) & inc_band.isin(['q1_lowest', 'q2_low'])
fp_eligible_idx = gdf_focus.index[fp_eligible]
n_fp  = min(PROGRAMME_HOMES, len(fp_eligible_idx))
fp_idx = cons_raw.loc[fp_eligible_idx].nlargest(n_fp).index  # rank by consumption within pool

# hp_composite: both actors coordinate, score all homes on three dimensions
composite_idx = composite_score.nlargest(PROGRAMME_HOMES).index

POLICY_MASKS = {
    'hp_top_users':    pd.Series(gdf_focus.index.isin(top_users_idx),  index=gdf_focus.index),
    'hp_fuel_poverty': pd.Series(gdf_focus.index.isin(fp_idx),         index=gdf_focus.index),
    'hp_composite':    pd.Series(gdf_focus.index.isin(composite_idx),  index=gdf_focus.index),
}

# Scenario design summary table
EPC_BAND_LABEL = {1: 'G', 2: 'F', 3: 'E', 4: 'D', 5: 'C', 6: 'B', 7: 'A'}
design_rows = []
for s in SCENARIOS:
    treated_idx = gdf_focus.index[POLICY_MASKS[s]]
    n = len(treated_idx)
    design_rows.append({
        'scenario':                s,
        'actor':                   SCENARIO_LABELS[s].replace('\n', ' '),
        'homes_treated':           n,
        'pct_stock':               f'{n / len(gdf_focus) * 100:.1f}%',
        'mean_epc_band':           EPC_BAND_LABEL.get(int(epc_ord.loc[treated_idx].mean().round()), '?'),
        'pct_efg_epc':             f'{epc_ord.loc[treated_idx].le(3).mean()*100:.1f}%',
        'pct_q1q2_income':         f'{inc_band.loc[treated_idx].isin(["q1_lowest", "q2_low"]).mean()*100:.1f}%',
        'mean_cons_kwh_yr':        f'{cons_raw.loc[treated_idx].mean():,.0f}',
    })

design_df = pd.DataFrame(design_rows)

if n_fp < PROGRAMME_HOMES:
    print(f'⚠ hp_fuel_poverty eligible pool ({len(fp_eligible_idx):,}) < PROGRAMME_HOMES ({PROGRAMME_HOMES:,})')
    print(f'  All {n_fp:,} eligible homes treated; consider lowering PROGRAMME_HOMES.')
else:
    print(f'hp_fuel_poverty eligible pool: {len(fp_eligible_idx):,} → treating {n_fp:,}')

print()
design_df

## 4) Cache LSOA input parquets

The parallel runner reads one parquet file per LSOA shard rather than subsetting a large
GeoDataFrame inside each worker process. This avoids pickling the full GeoDataFrame across
process boundaries and makes re-runs fast (parquets are only written if they don't exist).

File naming convention: `{PROCESS_COLUMN}={unit_val}.parquet` — same pattern as
`policy_scenarios_summary.ipynb` so outputs are interchangeable.

In [ ]:
UNIT_INPUT_DIR = OUTDIR / 'unit_inputs'
UNIT_INPUT_DIR.mkdir(parents=True, exist_ok=True)

written = 0
for unit_val in selected_units:
    dest = UNIT_INPUT_DIR / f'{PROCESS_COLUMN}={unit_val}.parquet'
    if not dest.exists():
        g_unit = gdf_focus[gdf_focus[PROCESS_COLUMN].astype(str) == str(unit_val)].copy()
        g_unit.to_parquet(dest, index=False)
        written += 1

print(f'Unit parquets: {len(selected_units)} total, {written} newly written → {UNIT_INPUT_DIR}')

## 5) Parallel runner functions

### Design pattern

The runner uses a **two-pass, treated-homes-only** approach:

1. **Baseline pass** (`_run_unit_baseline`): simulate all homes in an LSOA with
   `heatpump_adoption_rate=0`. Records energy use per home for the full window.

2. **Policy pass** (`_run_unit_scenario`): simulate *only* the treated homes in each LSOA,
   with `heatpump_adoption_rate=1.0` (every home in this sub-run receives a heat pump).
   Untreated homes are not re-simulated — their energy use is filled from the baseline at
   merge time. This avoids wasting compute on homes whose behaviour doesn't change.

   The `POLICY_MASKS` dict (built in Section 3) tells the runner which homes to include
   in each policy pass. Since the masks are pre-computed at city level, no within-LSOA
   sampling is needed.

### Seeding

Each (scenario, LSOA) combination gets a deterministic seed derived from its inputs.
This means re-runs produce identical results and scenario comparisons are not confounded
by stochastic variation across runs.

### Config override mechanism

`EnergyModel` accepts a `config_path` argument pointing to a YAML file that overrides
defaults. Each runner call writes a minimal override dict to a temp file, runs the model,
and the file is discarded. This keeps the baseline config on disk untouched.

In [ ]:
def _seed_from(*parts) -> int:
    key = '|'.join(map(str, parts)).encode()
    return int.from_bytes(hashlib.sha256(key).digest()[:4], 'big')


def _run_model_household_window(
    gdf_in: gpd.GeoDataFrame, *, start_utc: str, hours: int, cfg: dict | None
) -> pd.DataFrame:
    """Run EnergyModel for `hours` steps; return DataFrame of (AgentID, kwh_window)."""
    merged = _deep_merge(CALIBRATED_CFG, cfg or {})
    with tempfile.NamedTemporaryFile('w', suffix='.yaml', delete=False) as tmp:
        yaml.safe_dump(merged, tmp)
        cfg_path = tmp.name

    m = EnergyModel(
        gdf=gdf_in,
        climate_parquet=str(CLIMATE),
        climate_start=start_utc,
        collect_agent_level=False,
        agent_collect_every=168,
        config_path=cfg_path,
    )
    for _ in range(int(hours)):
        m.step()

    rows = []
    for h in m.household_agents:
        aid    = str(getattr(h, 'unique_id', ''))
        by_yr  = getattr(h, 'annual_kwh_by_year', {}) or {}
        rows.append({'AgentID': aid, 'kwh_window': float(sum(by_yr.values()))})

    out = pd.DataFrame(rows)
    if out.empty:
        out = pd.DataFrame({'AgentID': gdf_in['UPRN'].astype(str), 'kwh_window': 0.0})
    out['AgentID'] = out['AgentID'].astype(str)
    return out


def _run_unit_baseline(unit_val: str) -> pd.DataFrame:
    g = gpd.read_parquet(UNIT_INPUT_DIR / f'{PROCESS_COLUMN}={unit_val}.parquet')
    g['AgentID'] = g['UPRN'].astype(str)
    seed = _seed_from('baseline', unit_val)
    np.random.seed(seed); random.seed(seed)

    cfg = {'meta': {'name': 'baseline'}, 'model': {'heatpump_adoption_rate': 0.0}}
    res = _run_model_household_window(g, start_utc=START_UTC, hours=WINDOW_HOURS, cfg=cfg)

    keep = [c for c in META_COLS if c in g.columns]
    out  = g[keep].copy()
    out['AgentID'] = out['AgentID'].astype(str)
    out  = out.merge(res.rename(columns={'kwh_window': 'baseline_kwh_window'}), on='AgentID', how='left')
    out[PROCESS_COLUMN] = str(unit_val)
    return out


def _run_unit_scenario(unit_val: str, scenario: str) -> pd.DataFrame:
    g = gpd.read_parquet(UNIT_INPUT_DIR / f'{PROCESS_COLUMN}={unit_val}.parquet')
    g['AgentID'] = g['UPRN'].astype(str)
    seed = _seed_from('policy', unit_val, scenario)
    np.random.seed(seed); random.seed(seed)

    # Identify treated homes in this LSOA
    policy_mask_all = POLICY_MASKS[scenario]
    m_unit     = gdf_focus[PROCESS_COLUMN].astype(str).eq(str(unit_val))
    policy_ids = set(gdf_focus.loc[m_unit & policy_mask_all.fillna(False), 'AgentID'].astype(str))

    cols = ['AgentID', 'in_cohort', 'policy_kwh_window', 'scenario', PROCESS_COLUMN]
    if not policy_ids:
        return pd.DataFrame(columns=cols)

    # Only simulate treated homes; untreated homes filled from baseline at merge time
    g_policy = g[g['AgentID'].isin(policy_ids)].copy()
    g_policy['is_heatpump_candidate']     = 1
    g_policy['heatpump_candidate_class']  = 'priority'

    cfg = {'meta': {'name': scenario}, 'model': {'heatpump_adoption_rate': 1.0}}
    res = _run_model_household_window(g_policy, start_utc=START_UTC, hours=WINDOW_HOURS, cfg=cfg)

    cohort_df = g_policy[['AgentID']].copy()
    cohort_df['in_cohort'] = True
    out = cohort_df.merge(res.rename(columns={'kwh_window': 'policy_kwh_window'}), on='AgentID', how='left')
    out['AgentID']         = out['AgentID'].astype(str)
    out['scenario']        = scenario
    out[PROCESS_COLUMN]    = str(unit_val)
    return out


def _run_parallel(func, tasks, n_procs: int):
    """Run func over tasks using fork-based multiprocessing; fall back to serial on failure."""
    if n_procs <= 1:
        return [func(*t) if isinstance(t, tuple) else func(t) for t in tasks]
    try:
        ctx = mp.get_context('fork')
        with ctx.Pool(processes=n_procs) as pool:
            return pool.starmap(func, tasks) if isinstance(tasks[0], tuple) else pool.map(func, tasks)
    except Exception as e:
        print(f'Parallel pool failed ({e}); falling back to serial.')
        return [func(*t) if isinstance(t, tuple) else func(t) for t in tasks]

## 6) Run baseline + three scenarios (parallel)

**Execution order:**
1. Baseline: all 185 LSOAs × 1 = 185 tasks
2. Policy: 185 LSOAs × 3 scenarios = 555 tasks

**Merge logic:** for each scenario, every dwelling gets a `baseline_kwh_window` from the
baseline pass. Treated homes additionally get a `policy_kwh_window` from the policy pass;
untreated homes have `policy_kwh_window` filled from `baseline_kwh_window` (no change).
Annual figures are then computed by multiplying the window total by `ANNUAL_FACTOR`.

**`saving_kwh_year`** is baseline minus policy (positive = energy saved).
**`saving_gbp_year`** assumes 24p/kWh, roughly the 2023/24 Ofgem default tariff for
dual-fuel households — a deliberately conservative figure for the bill-saving estimates.

In [ ]:
LONG_PARQUET = OUTDIR / 'scenario_long.parquet'
LONG_CSV     = OUTDIR / 'scenario_long.csv'
SUMMARY_CSV  = OUTDIR / 'scenario_summary.csv'

baseline_tasks = [str(u) for u in selected_units]
scenario_tasks = [(str(u), s) for u in selected_units for s in SCENARIOS]

print(f'Baseline tasks:  {len(baseline_tasks)}')
print(f'Policy tasks:    {len(scenario_tasks)}')
print('Running baseline...')

baseline_rows = _run_parallel(_run_unit_baseline, baseline_tasks, n_procs=N_PROCS)
baseline_df   = pd.concat(baseline_rows, ignore_index=True)

print('Running policy scenarios...')
policy_rows = _run_parallel(_run_unit_scenario, scenario_tasks, n_procs=N_PROCS)
policy_df   = pd.concat(policy_rows, ignore_index=True)

# Merge: fill untreated homes from baseline, annualize, compute savings
keep_base = [c for c in [
    'AgentID', 'UPRN', AREA_COLUMN, PROCESS_COLUMN, 'geometry',
    'property_type', 'tenure', 'hh_income_band', 'schedule_type',
    'sap_band_ord', 'energy_cal_kwh', 'baseline_kwh_window',
] if c in baseline_df.columns]

scenario_frames = []
for s in SCENARIOS:
    base_s = baseline_df[keep_base].copy()
    base_s['scenario'] = s
    pol_s  = policy_df[policy_df['scenario'] == s][['AgentID', 'in_cohort', 'policy_kwh_window']].copy()
    merged = base_s.merge(pol_s, on='AgentID', how='left')
    merged['in_cohort']          = merged['in_cohort'].fillna(False).astype(bool)
    merged['policy_kwh_window']  = merged['policy_kwh_window'].fillna(merged['baseline_kwh_window'])
    scenario_frames.append(merged)

scenario_long = pd.concat(scenario_frames, ignore_index=True)
scenario_long['baseline_kwh_year'] = scenario_long['baseline_kwh_window'] * ANNUAL_FACTOR
scenario_long['policy_kwh_year']   = scenario_long['policy_kwh_window']   * ANNUAL_FACTOR
scenario_long['saving_kwh_year']   = scenario_long['baseline_kwh_year'] - scenario_long['policy_kwh_year']
scenario_long['saving_gbp_year']   = scenario_long['saving_kwh_year'] * 0.24
scenario_long['effective_adopter'] = (
    scenario_long['in_cohort'] & scenario_long['saving_kwh_year'].fillna(0).abs().gt(1e-9)
)

scenario_long.drop(columns=['geometry'], errors='ignore').to_parquet(LONG_PARQUET, index=False)
scenario_long.drop(columns=['geometry'], errors='ignore').to_csv(LONG_CSV, index=False)
print(f'Saved: {LONG_PARQUET}')
print(f'Rows: {len(scenario_long):,} ({len(scenario_long) // len(SCENARIOS):,} dwellings × {len(SCENARIOS)} scenarios)')

## 7) Summary table

Aggregates results to scenario level. Key columns:

- **`total_saving_gwh_yr`** — citywide annual demand reduction (the efficiency metric)
- **`mean_saving_eff_kwh`** — mean saving per home that actually adopted (how much each
  treated home benefits on average; higher for `hp_top_users` since those homes have the
  largest baseline demand)
- **`programme_cost_gbp_m`** — total programme cost in £m (same for all scenarios since
  `PROGRAMME_HOMES` is fixed; varies only if the fuel poverty pool is smaller)
- **`cost_per_gwh_gbp_m`** — £m per GWh saved (the value-for-money metric)
- **`equity_share_q1q2`** — fraction of total savings received by bottom 2 income quintiles
  (computed in Section 11)

In [ ]:
summary = (
    scenario_long.groupby('scenario', as_index=False)
    .agg(
        n_dwellings        = ('AgentID',          'nunique'),
        homes_treated      = ('in_cohort',         'sum'),
        effective_adopters = ('effective_adopter', 'sum'),
        total_saving_kwh   = ('saving_kwh_year',   'sum'),
    )
)

eff_stats = (
    scenario_long[scenario_long['effective_adopter']]
    .groupby('scenario', as_index=False)
    .agg(
        mean_saving_eff_kwh = ('saving_kwh_year', 'mean'),
        med_saving_eff_kwh  = ('saving_kwh_year', 'median'),
    )
)
summary = summary.merge(eff_stats, on='scenario', how='left')
summary['total_saving_gwh_yr']  = summary['total_saving_kwh'] / 1e6
summary['programme_cost_gbp_m'] = summary['homes_treated'] * HP_COST_GBP / 1e6
summary['cost_per_gwh_gbp_m']   = (summary['programme_cost_gbp_m']
                                    / summary['total_saving_gwh_yr'].clip(lower=1e-6))

summary.to_csv(SUMMARY_CSV, index=False)
summary[[
    'scenario', 'homes_treated', 'effective_adopters',
    'total_saving_gwh_yr', 'mean_saving_eff_kwh',
    'programme_cost_gbp_m', 'cost_per_gwh_gbp_m',
]]

## 8) Figure 1 — Aggregate demand reduction and cost-effectiveness

**Panel A** shows total GWh/yr saved under each scenario — the system-level efficiency
metric. `hp_top_users` should show the highest aggregate saving because it selects the
homes with the largest baseline demand (and therefore the largest absolute reduction
when a heat pump replaces a gas boiler).

**Panel B** shows cost per GWh saved (£m/GWh). Since all scenarios have the same number
of installations and the same cost assumption, this ratio is entirely driven by aggregate
demand reduction: the scenario that saves more GWh has lower cost-per-GWh. Panel B is
therefore the inverse of Panel A, but expressed in policy-legible terms for the paper.

**Paper use:** These two panels are Figure 3 in the PSJ draft. The key message is that
`hp_top_users` dominates on raw efficiency but `hp_fuel_poverty` and `hp_composite` trail
by only X GWh/yr — and the distributional figures (below) show who bears that cost.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
x       = np.arange(len(SCENARIOS))
colours = [SCENARIO_COLOURS[s] for s in SCENARIOS]
labels  = [SCENARIO_LABELS[s] for s in SCENARIOS]

# Panel A: aggregate GWh/yr
ax     = axes[0]
vals_a = [float(summary.loc[summary['scenario'] == s, 'total_saving_gwh_yr'].iloc[0]) for s in SCENARIOS]
bars   = ax.bar(x, vals_a, color=colours, width=0.55, edgecolor='white', linewidth=0.8)
for bar, v in zip(bars, vals_a):
    ax.text(bar.get_x() + bar.get_width() / 2, v + max(vals_a) * 0.01,
            f'{v:.2f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Annual demand reduction (GWh/yr)')
ax.set_title('A. System-level saving', fontweight='bold', fontsize=11)
ax.set_ylim(0, max(vals_a) * 1.20)

# Panel B: cost per GWh (same programme cost; driven by demand reduction)
ax     = axes[1]
vals_b = [float(summary.loc[summary['scenario'] == s, 'cost_per_gwh_gbp_m'].iloc[0]) for s in SCENARIOS]
bars   = ax.bar(x, vals_b, color=colours, width=0.55, edgecolor='white', linewidth=0.8)
for bar, v in zip(bars, vals_b):
    ax.text(bar.get_x() + bar.get_width() / 2, v + max(vals_b) * 0.01,
            f'£{v:.1f}m', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Programme cost per GWh saved (£m/GWh)')
ax.set_title('B. Cost-effectiveness', fontweight='bold', fontsize=11)
ax.set_ylim(0, max(vals_b) * 1.20)

legend_handles = [
    mpatches.Patch(color=SCENARIO_COLOURS[s], label=SCENARIO_LABELS[s].replace('\n', ' '))
    for s in SCENARIOS
]
fig.legend(handles=legend_handles, loc='lower center', ncol=3, frameon=False,
           fontsize=9, bbox_to_anchor=(0.5, -0.06))
fig.suptitle(
    f'Heat-pump targeting: same budget ({PROGRAMME_HOMES:,} homes, '
    f'£{PROGRAMME_HOMES * HP_COST_GBP / 1e6:.1f}m), different outcomes',
    fontsize=12, fontweight='bold', y=1.02,
)
plt.tight_layout()
fig.savefig(OUTDIR / 'figure1_aggregate_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved figure1_aggregate_comparison.png')

## 9) Figure 2 — Income quintile distribution of savings

This is the equity diagnostic and the paper's most politically legible figure.

**Panel A** (stacked bars) shows what share of total citywide savings goes to each income
quintile under each scenario. A programme is equity-enhancing if a larger share of the
aggregate saving lands with Q1/Q2 households. `hp_fuel_poverty` should show the most
bottom-weighted distribution; `hp_top_users` may benefit higher-income groups if high
consumption correlates with income (larger homes, more appliances).

**Panel B** (grouped bars) shows mean saving *per treated home* by income quintile and
scenario. This separates two effects:
- **Coverage effect**: how many Q1/Q2 homes are in the treated cohort?
- **Intensity effect**: how much does each treated Q1/Q2 home save?

If `hp_fuel_poverty` treats more low-income homes but each saves less (smaller, less
energy-hungry homes), Panel B will show that. The paper should discuss whether coverage
or intensity dominates the equity advantage.

In [ ]:
INCOME_ORDER   = ['q1_lowest', 'q2_low', 'q3_mid', 'q4_high', 'q5_highest']
INCOME_LABELS  = ['Q1 lowest', 'Q2 low', 'Q3 mid', 'Q4 high', 'Q5 highest']
INCOME_PALETTE = ['#2c5f8a', '#4c78a8', '#9bb9d4', '#e8b87a', '#d45f3c']

inc_df = (
    scenario_long[scenario_long['in_cohort']]
    .assign(income_band=lambda d: pd.Categorical(
        d['hh_income_band'].where(d['hh_income_band'].isin(INCOME_ORDER), 'unknown'),
        categories=INCOME_ORDER + ['unknown'],
    ))
    .groupby(['scenario', 'income_band'], observed=True)['saving_kwh_year']
    .sum().reset_index()
)
inc_pivot = inc_df.pivot(index='scenario', columns='income_band', values='saving_kwh_year').fillna(0)
inc_pivot = inc_pivot[[c for c in INCOME_ORDER if c in inc_pivot.columns]]
inc_share = inc_pivot.div(inc_pivot.sum(axis=1), axis=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Panel A: share of savings by income band
ax     = axes[0]
bottom = np.zeros(len(SCENARIOS))
x      = np.arange(len(SCENARIOS))
valid_bands = [c for c in INCOME_ORDER if c in inc_share.columns]
for i, band in enumerate(valid_bands):
    vals = [float(inc_share.loc[s, band]) if s in inc_share.index else 0.0 for s in SCENARIOS]
    ax.bar(x, vals, bottom=bottom, label=INCOME_LABELS[INCOME_ORDER.index(band)],
           color=INCOME_PALETTE[INCOME_ORDER.index(band)], width=0.55, edgecolor='white', linewidth=0.6)
    bottom += np.array(vals)
ax.set_xticks(x)
ax.set_xticklabels([SCENARIO_LABELS[s] for s in SCENARIOS], fontsize=9)
ax.set_ylabel('Share of total savings')
ax.set_title('A. Distribution of savings by income quintile', fontweight='bold', fontsize=11)
ax.set_ylim(0, 1.02)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.legend(title='Income band', loc='upper right', fontsize=8, frameon=False)

# Panel B: mean saving per treated home by income quintile
ax    = axes[1]
width = 0.25
x_b   = np.arange(len(valid_bands))
for i, s in enumerate(SCENARIOS):
    sub = (
        scenario_long[
            scenario_long['scenario'].eq(s)
            & scenario_long['effective_adopter']
            & scenario_long['hh_income_band'].isin(INCOME_ORDER)
        ]
        .groupby('hh_income_band')['saving_kwh_year'].mean()
        .reindex(valid_bands)
    )
    ax.bar(x_b + i * width, sub.values, width=width,
           color=SCENARIO_COLOURS[s], label=SCENARIO_LABELS[s].replace('\n', ' '),
           edgecolor='white', linewidth=0.6)
ax.set_xticks(x_b + width)
ax.set_xticklabels([INCOME_LABELS[INCOME_ORDER.index(c)] for c in valid_bands], fontsize=9)
ax.set_ylabel('Mean saving per treated home (kWh/yr)')
ax.set_title('B. Mean saving per treated home by income quintile', fontweight='bold', fontsize=11)
ax.legend(fontsize=8, frameon=False)

plt.tight_layout()
fig.savefig(OUTDIR / 'figure2_income_distribution.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved figure2_income_distribution.png')

## 10) Figure 3 — Targeting profile: who gets treated?

This figure makes the information-set argument visually concrete. For each scenario, it
shows the EPC band and income composition of the treated cohort as stacked bars — answering
the question: given that each actor can only see their own data, what kind of household
do they end up targeting?

Expected pattern:
- `hp_top_users` (energy supplier): skewed towards higher EPC bands (larger, better-insulated
  homes have higher absolute consumption even if they're efficient by area); income distribution
  may skew towards Q3–Q5 (wealthier households run more appliances).
- `hp_fuel_poverty` (local authority): strongly concentrated in EPC E/F/G and Q1/Q2 income
  by construction.
- `hp_composite` (coordinated): intermediate — EPC and income both weighted, so it finds
  homes that are both high-demand and vulnerable.

The divergence between the three profiles is the visual proof that information asymmetry
between actors is the mechanism producing the efficiency–equity tradeoff.

In [ ]:
EPC_BANDS   = [1, 2, 3, 4, 5, 6, 7]
EPC_LABELS  = ['G', 'F', 'E', 'D', 'C', 'B', 'A']
EPC_PALETTE = ['#b94040', '#d4643c', '#e8964c', '#f0c86e', '#7bbf7b', '#4c78a8', '#2c5f8a']

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

for col, s in enumerate(SCENARIOS):
    ax  = axes[col]
    ax2 = ax.twinx()

    treated_idx = gdf_focus.index[POLICY_MASKS[s]]
    cohort      = gdf_focus.loc[treated_idx]

    epc_counts  = (
        pd.to_numeric(cohort.get('sap_band_ord', pd.Series(4, index=cohort.index)), errors='coerce')
        .dropna().astype(int).value_counts().reindex(EPC_BANDS, fill_value=0)
    )
    epc_share   = epc_counts / epc_counts.sum()

    inc_counts  = (
        cohort.get('hh_income_band', pd.Series('', index=cohort.index))
        .value_counts().reindex(INCOME_ORDER, fill_value=0)
    )
    inc_share   = inc_counts / inc_counts.sum()

    # Left bar: EPC distribution
    bottom = 0.0
    for band_idx, (band, share) in enumerate(zip(EPC_BANDS, epc_share)):
        ax.bar(0, share, bottom=bottom, color=EPC_PALETTE[band_idx],
               edgecolor='white', linewidth=0.5, width=0.4)
        if share > 0.04:
            ax.text(0, bottom + share / 2, EPC_LABELS[band_idx],
                    ha='center', va='center', fontsize=8, color='white', fontweight='bold')
        bottom += share

    # Right bar: income distribution
    bottom = 0.0
    for band_idx, (band, share) in enumerate(zip(INCOME_ORDER, inc_share)):
        ax2.bar(1, share, bottom=bottom, color=INCOME_PALETTE[band_idx],
                edgecolor='white', linewidth=0.5, width=0.4)
        if share > 0.04:
            ax2.text(1, bottom + share / 2, f'Q{band_idx+1}',
                     ha='center', va='center', fontsize=8, color='white', fontweight='bold')
        bottom += share

    ax.set_xticks([0, 1])
    ax.set_xticklabels(['EPC band', 'Income band'], fontsize=9)
    ax.set_ylim(0, 1);  ax2.set_ylim(0, 1)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
    ax2.set_yticklabels([])
    ax.set_title(SCENARIO_LABELS[s], fontweight='bold',
                 color=SCENARIO_COLOURS[s], fontsize=10)
    ax.annotate(f'n={len(treated_idx):,}', xy=(0.5, 1.04), xycoords='axes fraction',
                ha='center', fontsize=8, color=GRAY_M)

axes[0].set_ylabel('Share of treated cohort')
fig.suptitle('Targeting profile: who gets treated under each scenario?',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(OUTDIR / 'figure3_targeting_profile.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved figure3_targeting_profile.png')

## 11) Figure 4 — Efficiency–equity frontier

The headline figure for Paper 2. Each scenario is plotted as a point in a 2D space where:
- **x-axis**: citywide annual demand reduction (GWh/yr) — efficiency
- **y-axis**: equity share — fraction of total savings received by Q1+Q2 households

The frontier makes the tradeoff explicit: a programme cannot simultaneously maximise both
dimensions unless it coordinates across institutional actors. The composite scenario
should sit above and to the right of both unilateral scenarios — or at minimum demonstrate
a superior equity share at comparable (if slightly lower) efficiency.

**Cost-per-GWh** is annotated below each point to connect the efficiency–equity tradeoff
to the budget language policymakers use.

**Paper argument:** The distance between `hp_composite` and the nearest unilateral scenario
is the quantified coordination dividend. If `hp_composite` is dominated by one of the
unilateral scenarios (i.e., it is neither more efficient nor more equitable), the composite
weight choice needs revisiting — this would be a meaningful finding in itself.

In [ ]:
frontier_rows = []
for s in SCENARIOS:
    sub          = scenario_long[scenario_long['scenario'] == s]
    total_saving = sub['saving_kwh_year'].sum()
    q1q2_saving  = sub[sub['hh_income_band'].isin(['q1_lowest', 'q2_low'])]['saving_kwh_year'].sum()
    frontier_rows.append({
        'scenario':          s,
        'total_saving_gwh':  float(summary.loc[summary['scenario'] == s, 'total_saving_gwh_yr'].iloc[0]),
        'equity_share_q1q2': q1q2_saving / total_saving if total_saving > 0 else 0.0,
        'cost_per_gwh':      float(summary.loc[summary['scenario'] == s, 'cost_per_gwh_gbp_m'].iloc[0]),
    })

frontier_df = pd.DataFrame(frontier_rows)

# Save equity share back to summary
summary = summary.merge(frontier_df[['scenario', 'equity_share_q1q2']], on='scenario', how='left')
summary.to_csv(SUMMARY_CSV, index=False)

fig, ax = plt.subplots(figsize=(7, 5))

for _, row in frontier_df.iterrows():
    s = row['scenario']
    ax.scatter(row['total_saving_gwh'], row['equity_share_q1q2'],
               s=220, color=SCENARIO_COLOURS[s], zorder=5, edgecolors='white', linewidths=1.2)
    ax.annotate(SCENARIO_LABELS[s],
                xy=(row['total_saving_gwh'], row['equity_share_q1q2']),
                xytext=(8, 5), textcoords='offset points',
                fontsize=9, color=SCENARIO_COLOURS[s], fontweight='bold')
    ax.annotate(f'£{row["cost_per_gwh"]:.1f}m/GWh',
                xy=(row['total_saving_gwh'], row['equity_share_q1q2']),
                xytext=(8, -12), textcoords='offset points',
                fontsize=8, color=GRAY_M)

ax.set_xlabel('Aggregate annual demand reduction (GWh/yr)', fontsize=11)
ax.set_ylabel('Equity share: savings to Q1+Q2 households', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
ax.set_title(
    f'Efficiency–equity frontier ({PROGRAMME_HOMES:,} installations, £{HP_COST_GBP/1e3:.0f}k/home)',
    fontsize=11, fontweight='bold',
)
plt.tight_layout()
fig.savefig(OUTDIR / 'figure4_efficiency_equity_frontier.png', dpi=200, bbox_inches='tight')
plt.show()

print('Efficiency–equity frontier:')
print(frontier_df[['scenario', 'total_saving_gwh', 'equity_share_q1q2', 'cost_per_gwh']]
      .to_string(index=False))

## 12) Export

Writes the full summary CSV and per-scenario cohort profile tables.
These are the machine-readable versions of the paper tables.

Files written to `notebooks/results/paper2_policy/`:
- `scenario_long.parquet` / `.csv` — dwelling-level results for all scenarios
- `scenario_summary.csv` — scenario-level aggregates with equity shares
- `{scenario}_epc_profile.csv` — EPC band distribution of treated cohort
- `{scenario}_income_profile.csv` — income band distribution of treated cohort
- `figure1_aggregate_comparison.png` — Paper 2 Figure 3
- `figure2_income_distribution.png` — Paper 2 Figure 4
- `figure3_targeting_profile.png` — Paper 2 supplementary
- `figure4_efficiency_equity_frontier.png` — Paper 2 headline figure

In [ ]:
EPC_BAND_LABEL = {1: 'G', 2: 'F', 3: 'E', 4: 'D', 5: 'C', 6: 'B', 7: 'A'}

for s in SCENARIOS:
    treated_idx = gdf_focus.index[POLICY_MASKS[s]]
    cohort      = gdf_focus.loc[treated_idx].copy()

    epc_profile = (
        pd.to_numeric(cohort.get('sap_band_ord', 4), errors='coerce').astype(int)
        .map(EPC_BAND_LABEL)
        .value_counts(normalize=True).rename('share').reset_index()
        .rename(columns={'index': 'epc_band'})
    )
    epc_profile.to_csv(OUTDIR / f'{s}_epc_profile.csv', index=False)

    inc_profile = (
        cohort.get('hh_income_band', pd.Series('', index=cohort.index))
        .value_counts(normalize=True).rename('share').reset_index()
        .rename(columns={'index': 'income_band'})
    )
    inc_profile.to_csv(OUTDIR / f'{s}_income_profile.csv', index=False)

print('All exports complete:')
for f in sorted(OUTDIR.glob('*.csv')) :
    print(f'  {f.name}')
for f in sorted(OUTDIR.glob('*.png')):
    print(f'  {f.name}')